# Notebook 01 - Kapruka Crawler

Verifies the Playwright crawler and programmatic fallback catalog.

**What this notebook checks:**
- Catalog file load and structure
- Allergen tag distribution
- Category breakdown
- Sample product inspection

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from config.settings import Settings
settings = Settings()
print(f'Catalog path: {settings.CATALOG_PATH}')

Catalog path: D:\Zuu Crew Agentic AI\Projects\Mini Project 03\kapruka-gift-concierge\data\catalog.json


In [2]:
# ── Option A: Run the live crawler (may hit anti-bot) ──
# import asyncio
# from src.crawler.kapruka_scraper import KaprukaScraper
# scraper = KaprukaScraper(settings)
# catalog = asyncio.run(scraper.scrape_all_categories())
# scraper.save_catalog(catalog, settings.CATALOG_PATH)
# print(f'Scraped {catalog.total_products} products')

# ── Option B: Programmatic fallback (recommended for dev) ──
from src.crawler.kapruka_scraper import KaprukaScraper, _build_programmatic_catalog, KaprukaProduct, ProductCatalog
from datetime import datetime

raw = _build_programmatic_catalog()
products = [KaprukaProduct(**r) for r in raw]
catalog = ProductCatalog(
    products=products,
    crawl_timestamp=datetime.now().isoformat(),
    total_products=len(products),
    categories_crawled=list(set(p.category.value for p in products)),
)

import json
with open(settings.CATALOG_PATH, 'w', encoding='utf-8') as f:
    json.dump(catalog.model_dump(), f, indent=2, ensure_ascii=False)

print(f'✅ Generated {catalog.total_products} products from fallback catalog')

✅ Generated 505 products from fallback catalog


In [3]:
# ── Category breakdown ──────────────────────────────────
from collections import Counter

categories = Counter(p.category.value for p in products)
print('\n📂 Category breakdown:')
for cat, count in categories.most_common():
    bar = '█' * (count // 5)
    print(f'  {cat:20s} {count:4d}  {bar}')


📂 Category breakdown:
  flowers                80  ████████████████
  electronics            80  ████████████████
  cakes                  75  ███████████████
  chocolates             60  ████████████
  gift-hampers           60  ████████████
  soft-toys              60  ████████████
  fruit-baskets          45  █████████
  greeting-cards         45  █████████


In [4]:
# ── Allergen distribution ───────────────────────────────
allergen_counts = Counter()
nut_free_count = 0
vegan_count = 0

for p in products:
    for a in p.contains_allergens:
        allergen_counts[a] += 1
    if 'nut-free' in p.tags:
        nut_free_count += 1
    if 'vegan' in p.tags:
        vegan_count += 1

print('\n⚠️ Allergen distribution:')
for allergen, count in allergen_counts.most_common():
    pct = count / len(products) * 100
    print(f'  {allergen:15s} {count:4d} products ({pct:.1f}%)')

print(f'\n✅ Nut-free products  : {nut_free_count} ({nut_free_count/len(products)*100:.1f}%)')
print(f'🌱 Vegan products     : {vegan_count} ({vegan_count/len(products)*100:.1f}%)')


⚠️ Allergen distribution:
  gluten           147 products (29.1%)
  dairy            135 products (26.7%)
  eggs             102 products (20.2%)
  soy               60 products (11.9%)
  nuts              57 products (11.3%)

✅ Nut-free products  : 448 (88.7%)
🌱 Vegan products     : 125 (24.8%)


In [5]:
# ── Spot check: first product in each category ──────────
by_cat = {}
for p in products:
    by_cat.setdefault(p.category.value, p)

print('\n🔍 Sample product per category:')
for cat, p in sorted(by_cat.items()):
    print(f'  [{cat}]')
    print(f'    Name     : {p.product_name}')
    print(f'    Price    : LKR {p.price_lkr:,.0f}')
    print(f'    Allergens: {p.contains_allergens or "None"}')
    print(f'    Tags     : {p.tags}')


🔍 Sample product per category:
  [cakes]
    Name     : Classic Chocolate Truffle Cake 0.5kg
    Price    : LKR 4,000
    Allergens: ['nuts', 'dairy', 'gluten', 'eggs']
    Tags     : []
  [chocolates]
    Name     : Cadbury Dairy Milk 100g Gift Box
    Price    : LKR 1,800
    Allergens: ['dairy', 'nuts', 'gluten', 'soy']
    Tags     : []
  [electronics]
    Name     : JBL Mini Bluetooth Speaker
    Price    : LKR 6,500
    Allergens: None
    Tags     : ['nut-free', 'dairy-free', 'gluten-free']
  [flowers]
    Name     : Red Rose Bouquet - 6 Stems
    Price    : LKR 2,800
    Allergens: None
    Tags     : ['nut-free', 'dairy-free', 'gluten-free', 'vegan']
  [fruit-baskets]
    Name     : Fresh Fruit Basket - Small (2kg) Size
    Price    : LKR 2,500
    Allergens: None
    Tags     : ['nut-free', 'dairy-free', 'gluten-free', 'vegan']
  [gift-hampers]
    Name     : Premium Ceylon Tea Hamper - Small Pack
    Price    : LKR 4,500
    Allergens: ['gluten']
    Tags     : ['nut-free',